# Forensic Analysis of the Original Pipeline

This notebook reproduces the original ProteinBERT + dual-PCA pipeline exactly as trained, evaluates it on the DeepACET independent test set (CPLM-sourced), and documents the three sources of metric inflation that explain the gap between the originally reported ~99% accuracy and the true cross-database AUC of 0.52.

**Requires:** `pos_pca_model_positive.pkl`, `neg_pca_model_negative.pkl`, `acetylation_model_1.pth`, `independent_test_positive.csv`, `independent_test_negative.csv`

In [ ]:
!git clone https://github.com/nadavbra/protein_bert.git -q
!cd protein_bert && git submodule init && git submodule update && python setup.py install -q
import sys
sys.path.append('/content/protein_bert')
print('ProteinBERT installed')

In [ ]:
import numpy as np
import pandas as pd
import pickle
import torch
from torch import nn
from sklearn.metrics import (accuracy_score, roc_auc_score,
    matthews_corrcoef, confusion_matrix, mean_squared_error)
from itertools import product as iproduct
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import files
uploaded = files.upload()
# Upload:
# pos_pca_model_positive.pkl
# neg_pca_model_negative.pkl
# acetylation_model_1.pth
# independent_test_positive.csv
# independent_test_negative.csv
# distinct_positive_clean.csv
# distinct_negative_clean.csv
# deepacet_training_pos.csv
# deepacet_training_neg.csv

In [ ]:
ind_pos = pd.read_csv('independent_test_positive.csv')
ind_neg = pd.read_csv('independent_test_negative.csv')
for df in [ind_pos, ind_neg]:
    if 'Sequence' in df.columns:
        df.rename(columns={'Sequence': 'sequence'}, inplace=True)

print(f'Test positive: {len(ind_pos)}')
print(f'Test negative: {len(ind_neg)}')
print(f'Sequence length: {ind_pos["sequence"].str.len().unique()}')

In [ ]:
from proteinbert import load_pretrained_model
from proteinbert.conv_and_global_attention_model import get_model_with_hidden_layers_as_outputs

seq_len = 33
batch_size = 64

pretrained_model_generator, input_encoder = load_pretrained_model()
bert_model = get_model_with_hidden_layers_as_outputs(
    pretrained_model_generator.create_model(seq_len))
print('ProteinBERT loaded')

In [ ]:
def adjust_sequences(seqs, target_len=31):
    corrected = []
    for seq in seqs:
        if not isinstance(seq, str) or len(seq.strip()) == 0:
            continue
        seq = seq.strip()[:target_len]
        seq += 'X' * (target_len - len(seq))
        corrected.append(seq)
    return corrected

def get_local_embeddings(sequences, seq_len=33, batch_size=64):
    all_local = []
    for i in range(0, len(sequences), batch_size):
        batch = sequences[i:i+batch_size]
        encoded = input_encoder.encode_X(batch, seq_len)
        local_repr, _ = bert_model.predict(encoded, batch_size=batch_size, verbose=0)
        all_local.append(local_repr.reshape(local_repr.shape[0], -1))
        if i % 640 == 0:
            print(f'  {i}/{len(sequences)}')
    result = np.vstack(all_local)
    print(f'Embeddings shape: {result.shape}')
    return result

pos_seqs = adjust_sequences(ind_pos['sequence'].tolist())
neg_seqs = adjust_sequences(ind_neg['sequence'].tolist())

print(f'Generating embeddings for {len(pos_seqs)} positive sequences...')
ind_pos_local = get_local_embeddings(pos_seqs)

print(f'\nGenerating embeddings for {len(neg_seqs)} negative sequences...')
ind_neg_local = get_local_embeddings(neg_seqs)

In [ ]:
def load_pca_bundle(path):
    with open(path, 'rb') as f:
        bundle = pickle.load(f)
    if isinstance(bundle, dict):
        return bundle.get('scaler'), bundle.get('pca')
    return None, None

scaler_pos, pca_pos = load_pca_bundle('pos_pca_model_positive.pkl')
scaler_neg, pca_neg = load_pca_bundle('neg_pca_model_negative.pkl')

print(f'Positive PCA: {pca_pos}')
print(f'Negative PCA: {pca_neg}')

In [ ]:
def apply_dual_pca(embeddings, scaler_pos, pca_pos, scaler_neg, pca_neg, label):
    reduced_list = []
    chosen = []
    for i, emb in enumerate(embeddings):
        emb_2d = emb.reshape(1, -1)
        e_pos = scaler_pos.transform(emb_2d) if scaler_pos else emb_2d
        r_pos = pca_pos.transform(e_pos)
        err_pos = mean_squared_error(e_pos, pca_pos.inverse_transform(r_pos))

        e_neg = scaler_neg.transform(emb_2d) if scaler_neg else emb_2d
        r_neg = pca_neg.transform(e_neg)
        err_neg = mean_squared_error(e_neg, pca_neg.inverse_transform(r_neg))

        if err_pos <= err_neg:
            reduced_list.append(r_pos.flatten())
            chosen.append('pos')
        else:
            reduced_list.append(r_neg.flatten())
            chosen.append('neg')

        if i % 500 == 0:
            print(f'  [{label}] {i}/{len(embeddings)}')

    reduced = np.array(reduced_list)
    n_pos = chosen.count('pos')
    print(f'  [{label}] chose pos PCA: {n_pos}/{len(embeddings)} ({n_pos/len(embeddings)*100:.1f}%)')
    print(f'  [{label}] chose neg PCA: {len(embeddings)-n_pos}/{len(embeddings)} ({(len(embeddings)-n_pos)/len(embeddings)*100:.1f}%)')
    return reduced, chosen

print('Applying dual-PCA to positive test sequences...')
ind_pos_reduced, pos_choices = apply_dual_pca(
    ind_pos_local, scaler_pos, pca_pos, scaler_neg, pca_neg, 'TEST_POS')

print('\nApplying dual-PCA to negative test sequences...')
ind_neg_reduced, neg_choices = apply_dual_pca(
    ind_neg_local, scaler_pos, pca_pos, scaler_neg, pca_neg, 'TEST_NEG')

In [ ]:
data_test  = np.vstack([ind_pos_reduced, ind_neg_reduced])
labels_test = np.array([1]*len(ind_pos_reduced) + [0]*len(ind_neg_reduced))

class OriginalMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_layers = nn.Sequential(
            nn.Linear(100, 64), nn.ReLU(),
            nn.Linear(64, 32),  nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return torch.sigmoid(self.linear_layers(x))

mlp = OriginalMLP()
mlp.load_state_dict(torch.load('acetylation_model_1.pth', map_location='cpu'))
mlp.eval()

probs = []
with torch.no_grad():
    for val in data_test:
        out = mlp(torch.tensor(val, dtype=torch.float32))
        probs.append(out.item())

probs = np.array(probs)
binary_preds = (probs >= 0.5).astype(int)

acc = accuracy_score(labels_test, binary_preds)
auc = roc_auc_score(labels_test, probs)
mcc = matthews_corrcoef(labels_test, binary_preds)
tn, fp, fn, tp = confusion_matrix(labels_test, binary_preds).ravel()
sn = tp / (tp + fn)
sp = tn / (tn + fp)

print('=' * 60)
print('  Original pipeline — DeepACET independent test')
print('=' * 60)
print(f'  Accuracy:    {acc:.4f}')
print(f'  AUC:         {auc:.4f}')
print(f'  MCC:         {mcc:.4f}')
print(f'  Sensitivity: {sn:.4f}')
print(f'  Specificity: {sp:.4f}')
print(f'  TP:{tp} TN:{tn} FP:{fp} FN:{fn}')
print('=' * 60)
print(f'\nMean predicted probability: {probs.mean():.4f}')
print(f'Predicted positive: {binary_preds.sum()}/{len(binary_preds)} ({binary_preds.mean()*100:.1f}%)')

In [ ]:
AA = 'ACDEFGHIKLMNPQRSTVWY'

def cksaap_encoding(sequence, k_max=3):
    seq = sequence.upper()
    features = []
    for k in range(k_max + 1):
        pair_counts = {a+b: 0 for a, b in iproduct(AA, AA)}
        total = 0
        for i in range(len(seq) - k - 1):
            pair = seq[i] + seq[i+k+1]
            if pair in pair_counts:
                pair_counts[pair] += 1
                total += 1
        features.extend([pair_counts[p]/total for p in pair_counts] if total > 0 else [0]*400)
    return features

In [ ]:
pos_clean = pd.read_csv('distinct_positive_clean.csv')
neg_clean = pd.read_csv('distinct_negative_clean.csv')

pos_feat = np.array([cksaap_encoding(s) for s in pos_clean['sequence']])
neg_feat = np.array([cksaap_encoding(s) for s in neg_clean['sequence']])

X = np.vstack([pos_feat, neg_feat])
y = np.array([1]*len(pos_feat) + [0]*len(neg_feat))

scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_aucs = []
for fold, (tr, val) in enumerate(skf.split(X_sc, y)):
    clf = MLPClassifier(hidden_layer_sizes=(128,64), activation='relu',
                         max_iter=200, random_state=42)
    clf.fit(X_sc[tr], y[tr])
    prob = clf.predict_proba(X_sc[val])[:,1]
    cv_aucs.append(roc_auc_score(y[val], prob))
    print(f'  Fold {fold+1}: AUC={cv_aucs[-1]:.4f}')
print(f'\nCV AUC: {np.mean(cv_aucs):.4f} ± {np.std(cv_aucs):.4f}')

In [ ]:
ind_pos_feat = np.array([cksaap_encoding(s) for s in ind_pos['sequence']])
ind_neg_feat = np.array([cksaap_encoding(s) for s in ind_neg['sequence']])
X_ind = np.vstack([ind_pos_feat, ind_neg_feat])
y_ind = np.array([1]*len(ind_pos_feat) + [0]*len(ind_neg_feat))

clf_final = MLPClassifier(hidden_layer_sizes=(128,64), activation='relu',
                            max_iter=200, random_state=42)
clf_final.fit(scaler.transform(X), y)

X_ind_sc   = scaler.transform(X_ind)
prob_ind   = clf_final.predict_proba(X_ind_sc)[:,1]
pred_ind   = clf_final.predict(X_ind_sc)

tn, fp, fn, tp = confusion_matrix(y_ind, pred_ind).ravel()
print('=== CKSAAP on DeepACET independent test ===')
print(f'Accuracy:    {accuracy_score(y_ind, pred_ind):.4f}')
print(f'AUC:         {roc_auc_score(y_ind, prob_ind):.4f}')
print(f'MCC:         {matthews_corrcoef(y_ind, pred_ind):.4f}')
print(f'Sensitivity: {tp/(tp+fn):.4f}')
print(f'Specificity: {tn/(tn+fp):.4f}')

In [ ]:
da_pos = pd.read_csv('deepacet_training_pos.csv')
da_neg = pd.read_csv('deepacet_training_neg.csv')
for df in [da_pos, da_neg]:
    if 'Sequence' in df.columns:
        df.rename(columns={'Sequence': 'sequence'}, inplace=True)

print(f'DeepACET train: {len(da_pos)} pos, {len(da_neg)} neg')

da_pos_feat = np.array([cksaap_encoding(s) for s in da_pos['sequence']])
da_neg_feat = np.array([cksaap_encoding(s) for s in da_neg['sequence']])
X_da = np.vstack([da_pos_feat, da_neg_feat])
y_da = np.array([1]*len(da_pos_feat) + [0]*len(da_neg_feat))

scaler_da = StandardScaler()
X_da_sc = scaler_da.fit_transform(X_da)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_aucs_da = []
for fold, (tr, val) in enumerate(skf.split(X_da_sc, y_da)):
    clf = MLPClassifier(hidden_layer_sizes=(128,64), activation='relu',
                         max_iter=200, random_state=42)
    clf.fit(X_da_sc[tr], y_da[tr])
    prob = clf.predict_proba(X_da_sc[val])[:,1]
    cv_aucs_da.append(roc_auc_score(y_da[val], prob))
    print(f'  Fold {fold+1}: AUC={cv_aucs_da[-1]:.4f}')
print(f'\nCV AUC: {np.mean(cv_aucs_da):.4f} ± {np.std(cv_aucs_da):.4f}')

clf_da = MLPClassifier(hidden_layer_sizes=(128,64), activation='relu',
                        max_iter=200, random_state=42)
clf_da.fit(X_da_sc, y_da)

X_ind_da = scaler_da.transform(X_ind)
prob_ind_da = clf_da.predict_proba(X_ind_da)[:,1]
pred_ind_da = clf_da.predict(X_ind_da)

tn, fp, fn, tp = confusion_matrix(y_ind, pred_ind_da).ravel()
print('\n=== CKSAAP trained on DeepACET data ===')
print(f'AUC: {roc_auc_score(y_ind, prob_ind_da):.4f}')
print(f'MCC: {matthews_corrcoef(y_ind, pred_ind_da):.4f}')
print(f'Sensitivity: {tp/(tp+fn):.4f}')
print(f'Specificity: {tn/(tn+fp):.4f}')

In [ ]:
def position_frequency_matrix(sequences, length=31):
    AA_list = list('ACDEFGHIKLMNPQRSTVWY')
    matrix = np.zeros((length, len(AA_list)))
    for seq in sequences:
        seq = seq.upper().strip()[:length].ljust(length, 'X')
        for pos, aa in enumerate(seq):
            if aa in AA_list:
                matrix[pos, AA_list.index(aa)] += 1
    row_sums = matrix.sum(axis=1, keepdims=True)
    return matrix / np.where(row_sums == 0, 1, row_sums)

pos_freq = position_frequency_matrix(pos_clean['sequence'].tolist())
neg_freq = position_frequency_matrix(neg_clean['sequence'].tolist())
diff = pos_freq - neg_freq

plt.figure(figsize=(14, 5))
plt.imshow(diff.T, aspect='auto', cmap='RdBu_r', vmin=-0.1, vmax=0.1)
plt.colorbar(label='Enrichment (acetylated minus non-acetylated)')
plt.xticks(range(31), list(range(-15, 16)), fontsize=7)
plt.yticks(range(20), list('ACDEFGHIKLMNPQRSTVWY'), fontsize=8)
plt.axvline(15, color='black', linewidth=2, linestyle='--')
plt.xlabel('Position relative to acetylated K')
plt.ylabel('Amino acid')
plt.title('Amino acid enrichment around acetylated lysines')
plt.tight_layout()
plt.savefig('sequence_enrichment.png', dpi=200)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
categories = ['Positive sequences', 'Negative sequences']
before = [72843, 72843]
after  = [27182, 7063]
x = np.arange(len(categories))
w = 0.35
ax.bar(x - w/2, before, w, label='Before CD-HIT', color='steelblue', alpha=0.8)
ax.bar(x + w/2, after,  w, label='After CD-HIT',  color='coral',     alpha=0.8)
for i, (b, a) in enumerate(zip(before, after)):
    pct = (b-a)/b*100
    ax.annotate(f'{pct:.0f}% removed', xy=(i+w/2, a),
                xytext=(0, 5), textcoords='offset points',
                ha='center', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel('Number of sequences')
ax.set_title('Sequence redundancy removed by CD-HIT (40% identity threshold)')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig('redundancy_figure.png', dpi=200)
plt.show()